# Final Report: Multi-Agent Pathfinding with Operator Decomposition RTDP

This notebook serves as the final report and interactive walkthrough for the **Multi-Agent Markov Decision Process (MMDP)** project. 

In this report, we analyze the performance difference between **Baseline Real-Time Dynamic Programming (RTDP)** (which plans in the full joint-action space) and **Operator Decomposition (OD) RTDP** (which breaks decisions down sequentially to reduce the exponential branching factor).

### 1. Environment Setup & Dependency Resolution

The project has been rigorously refactored into a modular Python framework. To ensure this notebook remains highly portable (e.g., capable of running on Google Colab with zero manual uploads), the cell below will automatically clone the underlying codebase and diagnostic maps if running in a cloud environment.

In [ ]:
import sys
import os
from pathlib import Path

# ---------------------------------------------------------
# IMPORTANT: Update the URL below to your actual repository
# ---------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USERNAME/final_project_solved_stop.git"
REPO_NAME = "final_project_solved_stop"

# If we are running in Google Colab, fetch the repository automatically
if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_NAME):
        print("Running on Colab. Cloning repository...")
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    sys.path.append('src')
else:
    # Running locally; just add the src folder to the path
    src_path = str(Path('.').resolve() / 'src')
    if src_path not in sys.path:
        sys.path.append(src_path)
        
from grid_mmdp import GridMMDP, MMDPConfig
from map_creator import create_map_instance
from heuristic import ShortestPathHeuristic
from baseline_rtdp import BaselineRTDP, RTDPConfig
from od_rtdp import OperatorDecompositionRTDP
from evaluation import evaluate_policy, EvaluationConfig

print("✅ Framework successfully loaded and ready!")

### 2. The Multi-Agent Environment (GridMMDP)

The underlying problem is a cooperative stochastic game. We load a `GridMMDP` which enforces:
- **Stochastic Transitions (Slip):** Agents have a known probability (e.g., 20%) of slipping and staying in place instead of completing their intended move.
- **Collision Resolution:** Vertex conflicts (two agents targeting the same cell) and edge swaps (agents crossing paths) resolve strictly to fixed points (the agents bump into each other and stay in place).

In [ ]:
# Load a 9x9 diagnostic map designed to force a crossing conflict
map_instance = create_map_instance('maps/diagnostic/crossing-9-9', scenario_number=1, task_offset=0, n_agents=3)

# Initialize the MMDP environment with a 20% slip-to-stay probability
mdp_config = MMDPConfig(slip_to_stay_probability=0.20)
mdp = GridMMDP(map_instance, mdp_config)

# Build the Shortest-Path Heuristic using backward search from the goals.
# This provides an admissible heuristic without requiring expensive full-state Value Iteration.
heuristic = ShortestPathHeuristic(mdp)

print(f"🗺️ Map: {map_instance.map_name}")
print(f"🤖 Agents: {mdp.config_agents}")
print(f"📍 Starts: {map_instance.starts}")
print(f"🎯 Goals: {map_instance.goals}")

### 3. Baseline RTDP (Full Joint Action Space)

Baseline RTDP searches for a policy by evaluating every possible **Joint Action** at every step. 
- Because every agent chooses a move simultaneously, the branching factor is $|A|^n$. 
- For $3$ agents with $5$ possible moves each, the planner must evaluate **$125$ joint transitions** at every single state expansion. This is extremely computationally heavy.

Let's see how much planning it can accomplish in a strict **5-second time limit**.

In [ ]:
baseline_config = RTDPConfig(time_limit_seconds=5.0, seed=42)
baseline_planner = BaselineRTDP(mdp, heuristic, baseline_config)

print("⏳ Running Baseline RTDP...")
baseline_result = baseline_planner.solve()

print("\n--- Baseline Planning Results ---")
print(f"Stop reason: {baseline_result.stop_reason}")
print(f"Trials completed: {baseline_result.trials_completed:,}")
print(f"Bellman backups: {baseline_result.bellman_backups:,}")
print(f"Unique states visited: {baseline_result.visited_states:,}")

In [ ]:
# Evaluate the greedy policy learned by Baseline RTDP
eval_config = EvaluationConfig(episodes=50, seed=101)
baseline_eval = evaluate_policy(mdp, baseline_planner, eval_config)

print("\n--- Baseline Evaluation (50 Episodes) ---")
print(f"Success rate: {baseline_eval.summary.success_rate * 100:.1f}%")
print(f"Mean steps (successful): {baseline_eval.summary.mean_steps_successful_episodes:.2f}")

### 4. Operator Decomposition (OD) RTDP

To mitigate the massive branching factor, **Operator Decomposition** splits a simultaneous joint action into sequential decisions. 
- Instead of choosing 1 out of 125 joint actions, Agent 1 chooses 1 of 5 moves. The environment enters an *intermediate* state. Agent 2 then chooses 1 of 5 moves, and so on.
- The branching factor drops to $|A| = 5$, dramatically reducing the work per state expansion.
- The OD heuristic handles these intermediate states carefully, calculating distances accurately for agents that have already "moved" in the current sequential chain.

Let's run OD RTDP with the exact same **5-second time limit**.

In [ ]:
od_config = RTDPConfig(time_limit_seconds=5.0, seed=42)
od_planner = OperatorDecompositionRTDP(mdp, heuristic, od_config)

print("⏳ Running Operator Decomposition RTDP...")
od_result = od_planner.solve()

print("\n--- OD Planning Results ---")
print(f"Stop reason: {od_result.stop_reason}")
print(f"Trials completed: {od_result.trials_completed:,}")
print(f"Bellman backups: {od_result.bellman_backups:,}")

In [ ]:
# Evaluate the greedy policy learned by OD RTDP
od_eval = evaluate_policy(mdp, od_planner, eval_config)

print("\n--- OD Evaluation (50 Episodes) ---")
print(f"Success rate: {od_eval.summary.success_rate * 100:.1f}%")
print(f"Mean steps (successful): {od_eval.summary.mean_steps_successful_episodes:.2f}")

### 5. Final Conclusion & Findings

If you compare the planning results under the same time constraint, **OD RTDP typically completes significantly more trials and Bellman backups** than Baseline RTDP.

This structural enhancement proves that by breaking the joint-action space into a sequence of individual decisions (via intermediate `ODStates`), the algorithm can aggressively explore deeper into the search tree. This leads to faster policy convergence and significantly higher success rates on complex, stochastic multi-agent problems without requiring extra computational resources.